-SCT213-C002-0141/2024 GACHOMO NJERU

-SCT213-C002-0063/2024 MWITA NYEHITA

-SCT213-C002-0142/2024 KARWEGA NJERU
# Sprint 1— Milestone 4: Statistical Inference & Analytical Modeling


**Objectives:**
- Perform hypothesis testing (t-test, chi-square, ANOVA)
- Correlation and regression analysis (simple & multiple)
- Time series / trend analysis across days and party sizes
- Interpret uncertainty and variability with confidence intervals
- Produce a full statistical evaluation and insight validation report

---
## 0. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, f_oneway, pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)

# ── Reproduce Sprint-2 processing pipeline ─────────────────────────────────────
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df_raw = pd.read_csv(url)
df = df_raw.copy()

df['tip_pct']         = (df['tip'] / df['total_bill'] * 100).round(2)
df['bill_per_person'] = (df['total_bill'] / df['size']).round(2)
day_avg               = df.groupby('day')['tip'].mean().round(2).reset_index()
day_avg.columns       = ['day', 'day_avg_tip']
df                    = pd.merge(df, day_avg, on='day', how='left')
df['tip_vs_day_avg']  = (df['tip'] - df['day_avg_tip']).round(2)
df['is_weekend']      = df['day'].isin(['Sat', 'Sun']).astype(int)

def tip_category(pct):
    if pct < 10:   return 'Low (<10%)'
    elif pct < 20: return 'Average (10-20%)'
    else:          return 'Generous (>20%)'

df['tip_category']   = df['tip_pct'].apply(tip_category)
df['sex_encoded']    = (df['sex']    == 'Male').astype(int)
df['smoker_encoded'] = (df['smoker'] == 'Yes').astype(int)
df['time_encoded']   = (df['time']   == 'Dinner').astype(int)
df['day_num']        = df['day'].map({'Thur': 1, 'Fri': 2, 'Sat': 3, 'Sun': 4})

# Global plot style
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.titlesize': 13, 'axes.labelsize': 11
})

print(f"Dataset ready | Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

---
## 1. Hypothesis Testing

Each test follows the same structured protocol:
> **H₀ (Null)** → **H₁ (Alternative)** → **Test chosen** → **α = 0.05** → **Decision** → **Interpretation**

---
### 1.1 Independent Samples t-Test — Do smokers tip differently from non-smokers?

In [ ]:
smokers     = df[df['smoker'] == 'Yes']['tip']
non_smokers = df[df['smoker'] == 'No']['tip']

# Levene's test for equal variances first
lev_stat, lev_p = stats.levene(smokers, non_smokers)
equal_var = lev_p > 0.05

t_stat, p_val = ttest_ind(smokers, non_smokers, equal_var=equal_var)
alpha = 0.05

# Effect size (Cohen's d)
pooled_std = np.sqrt((smokers.std()**2 + non_smokers.std()**2) / 2)
cohens_d   = (smokers.mean() - non_smokers.mean()) / pooled_std

# 95 % CI for the difference in means (Welch)
n1, n2 = len(smokers), len(non_smokers)
se_diff = np.sqrt(smokers.var()/n1 + non_smokers.var()/n2)
df_welch = (smokers.var()/n1 + non_smokers.var()/n2)**2 / \
           ((smokers.var()/n1)**2/(n1-1) + (non_smokers.var()/n2)**2/(n2-1))
t_crit = stats.t.ppf(0.975, df_welch)
diff_ci = (smokers.mean()-non_smokers.mean()-t_crit*se_diff,
           smokers.mean()-non_smokers.mean()+t_crit*se_diff)

print("="*60)
print("TEST 1 — Independent Samples t-Test")
print("H₀: Mean tip (smokers) = Mean tip (non-smokers)")
print("H₁: Mean tips differ between smokers and non-smokers")
print("-"*60)
print(f"  Smokers     : n={n1}, mean=${smokers.mean():.3f}, SD=${smokers.std():.3f}")
print(f"  Non-smokers : n={n2}, mean=${non_smokers.mean():.3f}, SD=${non_smokers.std():.3f}")
print(f"  Levene p    : {lev_p:.4f}  →  Equal variances assumed: {equal_var}")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_val:.4f}")
print(f"  Cohen's d   : {cohens_d:.4f}  (small < 0.2, medium < 0.5, large ≥ 0.8)")
print(f"  95% CI diff : ({diff_ci[0]:.3f}, {diff_ci[1]:.3f})")
print("-"*60)
if p_val < alpha:
    print(f"  DECISION: Reject H₀ (p={p_val:.4f} < α={alpha})")
    print("  INTERPRETATION: Significant difference in tipping between groups.")
else:
    print(f"  DECISION: Fail to reject H₀ (p={p_val:.4f} ≥ α={alpha})")
    print("  INTERPRETATION: No statistically significant difference in tipping")
    print("  between smokers and non-smokers. Any observed difference is likely")
    print("  due to chance. The CI includes 0, confirming no reliable effect.")
print("="*60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box + strip
sns.boxplot(data=df, x='smoker', y='tip',
            palette={'Yes':'#FF5722','No':'#2196F3'}, width=0.4, ax=axes[0])
sns.stripplot(data=df, x='smoker', y='tip',
              palette={'Yes':'#FF5722','No':'#2196F3'},
              alpha=0.35, jitter=True, size=4, ax=axes[0])
axes[0].set_title('Tip Distribution: Smoker vs Non-Smoker', fontweight='bold')
axes[0].set_xlabel('Smoker'); axes[0].set_ylabel('Tip ($)')
axes[0].text(0.97, 0.97, f'p = {p_val:.3f}\nd = {cohens_d:.3f}',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round', fc='white', ec='#ccc'))

# Confidence interval plot
means = [smokers.mean(), non_smokers.mean()]
cis   = [stats.t.interval(0.95, len(g)-1, loc=g.mean(), scale=stats.sem(g))
         for g in [smokers, non_smokers]]
labels = ['Smokers', 'Non-Smokers']
colors = ['#FF5722', '#2196F3']
for i, (m, ci, lbl, col) in enumerate(zip(means, cis, labels, colors)):
    axes[1].errorbar(i, m, yerr=[[m-ci[0]], [ci[1]-m]],
                     fmt='o', color=col, capsize=6, capthick=2,
                     markersize=9, label=lbl)
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(labels)
axes[1].set_title('Mean Tip ± 95% Confidence Interval', fontweight='bold')
axes[1].set_ylabel('Mean Tip ($)')
axes[1].set_xlim(-0.5, 1.5)

plt.tight_layout()
plt.savefig('m4_fig01_ttest.png', dpi=120, bbox_inches='tight')
plt.show()

---
### 1.2 t-Test — Do weekend visits produce higher tips than weekday visits?

In [ ]:
weekend  = df[df['is_weekend'] == 1]['tip']
weekday  = df[df['is_weekend'] == 0]['tip']

t2, p2   = ttest_ind(weekend, weekday, equal_var=False)
d2       = (weekend.mean() - weekday.mean()) / np.sqrt((weekend.std()**2 + weekday.std()**2)/2)

print("="*60)
print("TEST 2 — t-Test: Weekend vs Weekday Tips")
print("H₀: Mean tip (weekend) = Mean tip (weekday)")
print("H₁: Weekend tips > weekday tips")
print("-"*60)
print(f"  Weekend : n={len(weekend)}, mean=${weekend.mean():.3f}, SD=${weekend.std():.3f}")
print(f"  Weekday : n={len(weekday)}, mean=${weekday.mean():.3f}, SD=${weekday.std():.3f}")
print(f"  t       : {t2:.4f} | p-value : {p2:.4f} | Cohen's d : {d2:.4f}")
print("-"*60)
decision2 = 'Reject H₀' if p2 < 0.05 else 'Fail to reject H₀'
print(f"  DECISION: {decision2} (p={p2:.4f})")
if p2 < 0.05:
    print("  INTERPRETATION: Weekend tips are significantly higher.")
else:
    print("  INTERPRETATION: No statistically significant weekend premium.")
    print("  The observed difference could be due to sampling variation.")
print("="*60)

---
### 1.3 One-Way ANOVA — Does tip amount differ significantly across all four days?

In [ ]:
groups    = [df[df['day'] == d]['tip'].values for d in ['Thur','Fri','Sat','Sun']]
F, p_anov = f_oneway(*groups)

# Eta-squared effect size
grand_mean = df['tip'].mean()
ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in [pd.Series(g) for g in groups])
ss_total   = sum((df['tip'] - grand_mean)**2)
eta_sq     = ss_between / ss_total

print("="*60)
print("TEST 3 — One-Way ANOVA: Tip across Days")
print("H₀: Mean tips are equal across all four days")
print("H₁: At least one day has a different mean tip")
print("-"*60)
for day, g in zip(['Thur','Fri','Sat','Sun'], groups):
    print(f"  {day}: n={len(g)}, mean=${np.mean(g):.3f}, SD=${np.std(g):.3f}")
print(f"  F-statistic : {F:.4f}")
print(f"  p-value     : {p_anov:.4f}")
print(f"  η² (eta-sq) : {eta_sq:.4f}  (small<0.01, medium<0.06, large≥0.14)")
print("-"*60)
if p_anov < 0.05:
    print(f"  DECISION: Reject H₀ (p={p_anov:.4f} < 0.05)")
    print("  INTERPRETATION: Significant differences in tips across days.")
    print("  Post-hoc testing recommended to identify which pairs differ.")
else:
    print(f"  DECISION: Fail to reject H₀ (p={p_anov:.4f} ≥ 0.05)")
    print("  INTERPRETATION: No significant difference in mean tips across days.")
    print("  Day of the week alone is not a reliable predictor of tip amount.")
print("="*60)

In [ ]:
day_order = ['Thur','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(9, 4))
sns.violinplot(data=df, x='day', y='tip', order=day_order,
               palette='Blues', inner='quartile', ax=ax)
sns.swarmplot(data=df, x='day', y='tip', order=day_order,
              color='#333', size=3, alpha=0.5, ax=ax)
ax.set_title(f'Tip Distribution by Day  (ANOVA F={F:.2f}, p={p_anov:.3f}, η²={eta_sq:.3f})',
             fontweight='bold')
ax.set_xlabel('Day'); ax.set_ylabel('Tip ($)')
plt.tight_layout()
plt.savefig('m4_fig02_anova.png', dpi=120, bbox_inches='tight')
plt.show()

---
### 1.4 Chi-Square Test — Is tip generosity category associated with gender?

In [ ]:
ct = pd.crosstab(df['tip_category'], df['sex'])
chi2, p_chi, dof, expected = chi2_contingency(ct)

# Cramér's V effect size
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

print("="*60)
print("TEST 4 — Chi-Square Test of Independence")
print("H₀: Tip category and gender are independent")
print("H₁: Tip category and gender are associated")
print("-"*60)
print("Observed contingency table:")
print(ct.to_string())
print("\nExpected frequencies:")
print(pd.DataFrame(expected.round(1), index=ct.index, columns=ct.columns).to_string())
print(f"\n  χ² statistic : {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value      : {p_chi:.4f}")
print(f"  Cramér's V   : {cramers_v:.4f}  (weak<0.1, moderate<0.3, strong≥0.3)")
print("-"*60)
if p_chi < 0.05:
    print(f"  DECISION: Reject H₀ (p={p_chi:.4f} < 0.05)")
    print("  INTERPRETATION: Tip category is significantly associated with gender.")
else:
    print(f"  DECISION: Fail to reject H₀ (p={p_chi:.4f} ≥ 0.05)")
    print("  INTERPRETATION: No significant association between tip category")
    print("  and gender. Gender does not predict generosity category.")
print("="*60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Observed vs Expected
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', ax=axes[0],
            color=['#E91E63','#1976D2'], edgecolor='white')
axes[0].set_title('Tip Category by Gender (% within category)', fontweight='bold')
axes[0].set_xlabel('Tip Category')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='Gender', frameon=False)

# Heatmap of residuals
residuals = (ct.values - expected) / np.sqrt(expected)
sns.heatmap(pd.DataFrame(residuals, index=ct.index, columns=ct.columns),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=axes[1])
axes[1].set_title(f'Standardised Residuals  (χ²={chi2:.2f}, p={p_chi:.3f})', fontweight='bold')

plt.tight_layout()
plt.savefig('m4_fig03_chisq.png', dpi=120, bbox_inches='tight')
plt.show()

---
### 1.5 Chi-Square — Is smoking status associated with meal time (Lunch vs Dinner)?

In [ ]:
ct2 = pd.crosstab(df['smoker'], df['time'])
chi2b, p_chi2b, dof2, exp2 = chi2_contingency(ct2)
cv2 = np.sqrt(chi2b / (ct2.values.sum() * (min(ct2.shape) - 1)))

print("="*60)
print("TEST 5 — Chi-Square: Smoker Status vs Meal Time")
print("H₀: Smoking status and meal time are independent")
print("H₁: Smoking status and meal time are associated")
print("-"*60)
print(ct2)
print(f"\n  χ²={chi2b:.4f} | df={dof2} | p={p_chi2b:.4f} | Cramér's V={cv2:.4f}")
if p_chi2b < 0.05:
    print(f"  DECISION: Reject H₀ — meal time and smoking are associated.")
else:
    print(f"  DECISION: Fail to reject H₀ — meal time and smoking are independent.")
print("="*60)

---
## 2. Correlation Analysis

---
### 2.1 Pearson & Spearman Correlation Matrix

In [ ]:
numeric_cols = ['total_bill','tip','size','tip_pct',
                'bill_per_person','sex_encoded','smoker_encoded','time_encoded','is_weekend']

pearson_corr  = df[numeric_cols].corr(method='pearson').round(3)
spearman_corr = df[numeric_cols].corr(method='spearman').round(3)

print("Pearson Correlation Matrix (linear relationships):")
print(pearson_corr)
print("\nSpearman Correlation Matrix (monotonic relationships):")
print(spearman_corr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

for ax, corr, title in zip(axes,
    [pearson_corr, spearman_corr],
    ['Pearson (Linear)', 'Spearman (Monotonic)']):
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5,
                annot_kws={'size': 8}, ax=ax)
    ax.set_title(f'Correlation Matrix — {title}', fontweight='bold')

plt.tight_layout()
plt.savefig('m4_fig04_corr.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.2 Pairwise Significance Testing (tip vs key variables)

In [ ]:
predictors = ['total_bill','size','tip_pct','bill_per_person',
              'sex_encoded','smoker_encoded','time_encoded','is_weekend']

results = []
for col in predictors:
    r_p, p_p = pearsonr(df[col], df['tip'])
    r_s, p_s = spearmanr(df[col], df['tip'])
    sig = '***' if min(p_p, p_s) < 0.001 else '**' if min(p_p, p_s) < 0.01 else '*' if min(p_p, p_s) < 0.05 else 'ns'
    results.append({'Variable': col,
                    'Pearson r': round(r_p,3), 'Pearson p': round(p_p,4),
                    'Spearman ρ': round(r_s,3), 'Spearman p': round(p_s,4),
                    'Significance': sig})

corr_df = pd.DataFrame(results).sort_values('Pearson r', ascending=False, key=abs)
print("Correlation with 'tip' (sorted by |Pearson r|):")
print(corr_df.to_string(index=False))
print("\nSignificance codes: *** p<0.001 | ** p<0.01 | * p<0.05 | ns = not significant")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2196F3' if v > 0 else '#FF5722' for v in corr_df['Pearson r']]
bars = ax.barh(corr_df['Variable'], corr_df['Pearson r'], color=colors, edgecolor='white')

for bar, sig in zip(bars, corr_df['Significance']):
    x = bar.get_width()
    ax.text(x + (0.01 if x >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            sig, va='center', ha='left' if x >= 0 else 'right', fontsize=10, fontweight='bold')

ax.axvline(0, color='#333', lw=0.8)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.set_title('Pearson Correlation with Tip ($) — Significance annotated', fontweight='bold')
ax.set_xlim(-0.25, 0.75)

plt.tight_layout()
plt.savefig('m4_fig05_corrbar.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Regression Analysis

---
### 3.1 Simple Linear Regression — Total Bill → Tip

In [ ]:
from scipy.stats import t as t_dist

X_slr = df[['total_bill']].values
y_slr = df['tip'].values

model_slr = LinearRegression()
model_slr.fit(X_slr, y_slr)
y_pred_slr = model_slr.predict(X_slr)

n    = len(y_slr)
k    = 1
ss_res = np.sum((y_slr - y_pred_slr)**2)
ss_tot = np.sum((y_slr - y_slr.mean())**2)
r2_slr = 1 - ss_res/ss_tot
mse_slr = ss_res / (n - k - 1)
se_coef = np.sqrt(mse_slr / np.sum((X_slr - X_slr.mean())**2))
t_coef  = model_slr.coef_[0] / se_coef
p_coef  = 2 * t_dist.sf(abs(t_coef), df=n-2)
ci_low  = model_slr.coef_[0] - 1.96*se_coef
ci_high = model_slr.coef_[0] + 1.96*se_coef
rmse_slr = np.sqrt(mse_slr)

print("="*60)
print("SIMPLE LINEAR REGRESSION: tip ~ total_bill")
print("-"*60)
print(f"  Intercept   : {model_slr.intercept_:.4f}")
print(f"  Coefficient : {model_slr.coef_[0]:.4f}  (SE={se_coef:.4f})")
print(f"  t-statistic : {t_coef:.4f}")
print(f"  p-value     : {p_coef:.6f}")
print(f"  95% CI      : ({ci_low:.4f}, {ci_high:.4f})")
print(f"  R²          : {r2_slr:.4f}")
print(f"  RMSE        : {rmse_slr:.4f}")
print("-"*60)
print(f"  MODEL: tip = {model_slr.intercept_:.4f} + {model_slr.coef_[0]:.4f} × total_bill")
print(f"  INTERPRETATION: For every $1 increase in total_bill,")
print(f"  tip increases by ${model_slr.coef_[0]:.4f} on average.")
print(f"  The model explains {r2_slr*100:.1f}% of variance in tip.")
print("="*60)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter + fit
axes[0].scatter(df['total_bill'], df['tip'], alpha=0.5, s=40, color='#2196F3')
x_line = np.linspace(df['total_bill'].min(), df['total_bill'].max(), 300)
axes[0].plot(x_line, model_slr.intercept_ + model_slr.coef_[0]*x_line,
             color='#FF5722', lw=2)
axes[0].set_title(f'Fit: tip = {model_slr.intercept_:.2f} + {model_slr.coef_[0]:.2f}×bill\nR²={r2_slr:.3f}', fontweight='bold')
axes[0].set_xlabel('Total Bill ($)'); axes[0].set_ylabel('Tip ($)')

# Residual plot
residuals = y_slr - y_pred_slr
axes[1].scatter(y_pred_slr, residuals, alpha=0.5, s=40, color='#4CAF50')
axes[1].axhline(0, color='#FF5722', lw=1.5, ls='--')
axes[1].set_title('Residual Plot', fontweight='bold')
axes[1].set_xlabel('Fitted Values'); axes[1].set_ylabel('Residuals')

# Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals', fontweight='bold')
axes[2].get_lines()[1].set_color('#FF5722')

plt.tight_layout()
plt.savefig('m4_fig06_slr.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.2 Multiple Linear Regression — Full Feature Model

In [ ]:
features = ['total_bill','size','bill_per_person',
            'sex_encoded','smoker_encoded','time_encoded','is_weekend']

X_mlr = df[features]
y_mlr = df['tip']

X_train, X_test, y_train, y_test = train_test_split(X_mlr, y_mlr, test_size=0.25, random_state=42)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model_mlr = LinearRegression()
model_mlr.fit(X_train_s, y_train)

y_pred_train = model_mlr.predict(X_train_s)
y_pred_test  = model_mlr.predict(X_test_s)

r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test,  y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

# Adjusted R²
n_train = len(y_train)
p_feat  = len(features)
adj_r2  = 1 - (1-r2_train)*(n_train-1)/(n_train-p_feat-1)

print("="*60)
print("MULTIPLE LINEAR REGRESSION: tip ~ all features")
print("-"*60)
print(f"  Train R²    : {r2_train:.4f}  | Adjusted R²: {adj_r2:.4f}")
print(f"  Test R²     : {r2_test:.4f}")
print(f"  Train RMSE  : {rmse_train:.4f}")
print(f"  Test RMSE   : {rmse_test:.4f}")
print("-"*60)
print("  Standardised Coefficients (|value| = importance):")
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model_mlr.coef_})\
            .sort_values('Coefficient', key=abs, ascending=False)
for _, row in coef_df.iterrows():
    bar = '█' * int(abs(row['Coefficient'])*20)
    sign = '+' if row['Coefficient'] > 0 else '-'
    print(f"    {row['Feature']:20s}: {sign}{abs(row['Coefficient']):.4f}  {bar}")
print("-"*60)
print(f"  INTERPRETATION: total_bill dominates prediction. The model")
print(f"  explains {r2_test*100:.1f}% of tip variance on held-out data.")
print(f"  Demographic features (sex, smoker) add little predictive power.")
print("="*60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Feature importance
colors_coef = ['#2196F3' if c > 0 else '#FF5722' for c in coef_df['Coefficient']]
axes[0].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
axes[0].axvline(0, color='#333', lw=0.8)
axes[0].set_title('Standardised Coefficients\n(Multiple Regression)', fontweight='bold')
axes[0].set_xlabel('Standardised Coefficient')

# Actual vs Predicted
axes[1].scatter(y_test, y_pred_test, alpha=0.6, s=50, color='#4CAF50', edgecolors='white')
lims = [min(y_test.min(), y_pred_test.min())-0.5,
        max(y_test.max(), y_pred_test.max())+0.5]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
axes[1].set_xlabel('Actual Tip ($)')
axes[1].set_ylabel('Predicted Tip ($)')
axes[1].set_title(f'Actual vs Predicted (Test Set)\nR²={r2_test:.3f}, RMSE={rmse_test:.3f}', fontweight='bold')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('m4_fig07_mlr.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.3 Model Comparison — Simple vs Multiple Regression

In [ ]:
# Re-evaluate SLR on same test split for fair comparison
X_slr_train = X_train[['total_bill']]
X_slr_test  = X_test[['total_bill']]

model_slr2 = LinearRegression().fit(X_slr_train, y_train)
y_pred_slr_test = model_slr2.predict(X_slr_test)
r2_slr_test   = r2_score(y_test, y_pred_slr_test)
rmse_slr_test = np.sqrt(mean_squared_error(y_test, y_pred_slr_test))

comparison = pd.DataFrame({
    'Model'    : ['Simple LR (bill only)', 'Multiple LR (7 features)'],
    'Features' : [1, 7],
    'Test R²'  : [round(r2_slr_test,4), round(r2_test,4)],
    'Test RMSE': [round(rmse_slr_test,4), round(rmse_test,4)]
})

print("="*60)
print("MODEL COMPARISON")
print("-"*60)
print(comparison.to_string(index=False))
print("-"*60)
r2_gain = (r2_test - r2_slr_test) * 100
print(f"  R² gain from adding 6 features: +{r2_gain:.2f} percentage points")
print(f"  INTERPRETATION: The multiple regression barely improves over")
print(f"  simple regression, confirming that total_bill alone captures")
print(f"  most of the explainable variance in tip amount.")
print("="*60)

---
## 4. Trend Analysis

Although this dataset has no true time index, we can analyse ordered trends across:
- Days of the week (Thursday → Sunday, a sequential service window)
- Party size (an ordered discrete variable)

---
### 4.1 Day-Trend Analysis — Tips Across the Week

In [ ]:
day_trend = df.groupby(['day','day_num']).agg(
    n          =('tip','count'),
    mean_tip   =('tip','mean'),
    median_tip =('tip','median'),
    std_tip    =('tip','std'),
    mean_bill  =('total_bill','mean'),
    mean_pct   =('tip_pct','mean')
).reset_index().sort_values('day_num')

# Linear trend (Mann-Kendall-like: simple Pearson on day_num)
r_day, p_day = pearsonr(day_trend['day_num'], day_trend['mean_tip'])

print("Day-level trend summary:")
print(day_trend[['day','n','mean_tip','median_tip','std_tip','mean_pct']].to_string(index=False))
print(f"\nPearson r (day_num vs mean_tip) = {r_day:.4f}, p = {p_day:.4f}")
print("Interpretation: ", 
      "Significant linear trend across days." if p_day < 0.05
      else "No significant linear trend across days — tips are relatively stable.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Mean tip trend
axes[0].plot(day_trend['day'], day_trend['mean_tip'],
             marker='o', color='#2196F3', lw=2, markersize=9, label='Mean tip')
axes[0].fill_between(
    day_trend['day'],
    day_trend['mean_tip'] - day_trend['std_tip']/np.sqrt(day_trend['n']),
    day_trend['mean_tip'] + day_trend['std_tip']/np.sqrt(day_trend['n']),
    alpha=0.2, color='#2196F3', label='±SEM'
)
axes[0].plot(day_trend['day'], day_trend['median_tip'],
             marker='s', color='#FF5722', lw=1.5, ls='--', markersize=7, label='Median tip')
for _, row in day_trend.iterrows():
    axes[0].annotate(f"n={int(row['n'])}",
                     xy=(row['day'], row['mean_tip']),
                     xytext=(0, 12), textcoords='offset points',
                     ha='center', fontsize=8, color='#555')
axes[0].set_title('Tip Trend Across Days (Mean ± SEM)', fontweight='bold')
axes[0].set_xlabel('Day of Week'); axes[0].set_ylabel('Tip ($)')
axes[0].legend(frameon=False)

# Mean tip % trend
axes[1].bar(day_trend['day'], day_trend['mean_pct'],
            color=['#81D4FA','#29B6F6','#0288D1','#01579B'], edgecolor='white')
axes[1].set_title('Mean Tip % Trend Across Days', fontweight='bold')
axes[1].set_xlabel('Day of Week'); axes[1].set_ylabel('Mean Tip %')
axes[1].set_ylim(0, day_trend['mean_pct'].max() + 3)
for i, (_, row) in enumerate(day_trend.iterrows()):
    axes[1].text(i, row['mean_pct'] + 0.4, f"{row['mean_pct']:.1f}%",
                 ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('m4_fig08_daytrend.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.2 Party Size Trend — How tip behaviour changes with group size

In [ ]:
size_trend = df.groupby('size').agg(
    n          =('tip','count'),
    mean_tip   =('tip','mean'),
    mean_pct   =('tip_pct','mean'),
    mean_bill  =('total_bill','mean'),
    std_pct    =('tip_pct','std')
).reset_index()

r_size, p_size = pearsonr(size_trend['size'], size_trend['mean_pct'])

# Linear fit
m_s, b_s = np.polyfit(size_trend['size'], size_trend['mean_pct'], 1)

print("Party-size trend summary:")
print(size_trend.to_string(index=False))
print(f"\nPearson r (size vs mean_pct) = {r_size:.4f}, p = {p_size:.4f}")
print(f"Linear trend: tip_pct = {b_s:.2f} + {m_s:.2f} × size")
print("Interpretation:",
      "Significant downward trend — larger groups tip proportionally less."
      if p_size < 0.05 else "No significant trend.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Tip % vs party size with trend
axes[0].errorbar(size_trend['size'], size_trend['mean_pct'],
                 yerr=size_trend['std_pct']/np.sqrt(size_trend['n']),
                 fmt='o', color='#9C27B0', markersize=9, capsize=5, lw=2, label='Mean ± SEM')
x_fit = np.linspace(size_trend['size'].min(), size_trend['size'].max(), 100)
axes[0].plot(x_fit, b_s + m_s*x_fit, 'r--', lw=1.8,
             label=f'Trend (slope={m_s:.2f}%/person)')
axes[0].set_xticks(size_trend['size'])
axes[0].set_title(f'Tip % vs Party Size\nr={r_size:.3f}, p={p_size:.3f}', fontweight='bold')
axes[0].set_xlabel('Party Size'); axes[0].set_ylabel('Mean Tip %')
axes[0].legend(frameon=False)

# Total bill and tip by party size (dual axis)
ax2 = axes[1].twinx()
axes[1].bar(size_trend['size'] - 0.2, size_trend['mean_bill'],
            width=0.35, color='#81D4FA', label='Mean Bill ($)', alpha=0.9)
ax2.bar(size_trend['size'] + 0.2, size_trend['mean_tip'],
        width=0.35, color='#FF8A65', label='Mean Tip ($)', alpha=0.9)
axes[1].set_xlabel('Party Size')
axes[1].set_ylabel('Mean Total Bill ($)', color='#0288D1')
ax2.set_ylabel('Mean Tip ($)', color='#E64A19')
axes[1].set_title('Mean Bill & Tip by Party Size', fontweight='bold')
axes[1].set_xticks(size_trend['size'])
axes[1].legend(loc='upper left', frameon=False)
ax2.legend(loc='upper right', frameon=False)
ax2.spines['top'].set_visible(False)

plt.tight_layout()
plt.savefig('m4_fig09_sizetrend.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Uncertainty & Variability Analysis

---
### 5.1 Bootstrap Confidence Intervals — Mean Tip by Smoker Status

In [ ]:
def bootstrap_ci(data, stat_fn=np.mean, n_boot=5000, ci=95):
    boot_stats = [stat_fn(np.random.choice(data, size=len(data), replace=True))
                  for _ in range(n_boot)]
    lo = np.percentile(boot_stats, (100-ci)/2)
    hi = np.percentile(boot_stats, ci + (100-ci)/2)
    return np.array(boot_stats), lo, hi

groups_bs = {'Smoker': df[df['smoker']=='Yes']['tip'].values,
             'Non-Smoker': df[df['smoker']=='No']['tip'].values,
             'Weekend': df[df['is_weekend']==1]['tip'].values,
             'Weekday': df[df['is_weekend']==0]['tip'].values}

print("="*65)
print("BOOTSTRAP CONFIDENCE INTERVALS (5,000 resamples, 95% CI)")
print("-"*65)
bs_results = {}
for name, data in groups_bs.items():
    boot, lo, hi = bootstrap_ci(data)
    bs_results[name] = {'mean': data.mean(), 'lo': lo, 'hi': hi,
                        'width': hi-lo, 'boot': boot}
    print(f"  {name:15s}: mean=${data.mean():.3f} | 95% CI = (${lo:.3f}, ${hi:.3f}) | width=${hi-lo:.3f}")

print("-"*65)
print("  Narrower CI → less variability / more data → more certainty")
print("  Non-smokers have a tighter CI because there are more of them.")
print("="*65)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bootstrap distributions
colors_bs = ['#FF5722','#2196F3','#9C27B0','#4CAF50']
for i, (name, res) in enumerate(bs_results.items()):
    axes[0].hist(res['boot'], bins=50, alpha=0.55, color=colors_bs[i], label=name)
    axes[0].axvline(res['mean'], color=colors_bs[i], lw=2, ls='--')
axes[0].set_title('Bootstrap Distributions of Mean Tip', fontweight='bold')
axes[0].set_xlabel('Mean Tip ($)'); axes[0].set_ylabel('Frequency')
axes[0].legend(frameon=False, fontsize=8)

# CI comparison plot
names_bs = list(bs_results.keys())
means_bs = [bs_results[n]['mean'] for n in names_bs]
los      = [bs_results[n]['lo']   for n in names_bs]
his      = [bs_results[n]['hi']   for n in names_bs]
for i, (name, m, lo, hi, col) in enumerate(zip(names_bs, means_bs, los, his, colors_bs)):
    axes[1].plot([lo, hi], [i, i], color=col, lw=4, alpha=0.6)
    axes[1].plot(m, i, 'o', color=col, markersize=10, zorder=5)
    axes[1].text(hi + 0.02, i, f'${m:.2f}\n[{lo:.2f},{hi:.2f}]',
                 va='center', fontsize=8)
axes[1].set_yticks(range(len(names_bs)))
axes[1].set_yticklabels(names_bs)
axes[1].set_xlabel('Mean Tip ($)')
axes[1].set_title('95% Bootstrap CIs — Mean Tip by Group', fontweight='bold')
axes[1].set_xlim(1.8, 3.7)

plt.tight_layout()
plt.savefig('m4_fig10_bootstrap.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.2 Coefficient of Variation — Comparing Relative Variability Across Groups

In [ ]:
cv_df = df.groupby(['day','time'])['tip'].agg(
    n='count', mean='mean', std='std'
).reset_index()
cv_df['CV%']   = (cv_df['std'] / cv_df['mean'] * 100).round(1)
cv_df['SEM']   = (cv_df['std'] / np.sqrt(cv_df['n'])).round(4)
cv_df['CI_low']  = (cv_df['mean'] - 1.96*cv_df['SEM']).round(3)
cv_df['CI_high'] = (cv_df['mean'] + 1.96*cv_df['SEM']).round(3)
cv_df = cv_df.sort_values('CV%', ascending=False)

print("Variability Summary by Day × Time:")
print(cv_df[['day','time','n','mean','std','CV%','CI_low','CI_high']].to_string(index=False))
print("\nCV% = Coefficient of Variation. Higher = more unpredictable tipping.")
print("Groups with small n have wider CIs and higher uncertainty.")

---
## 6. Statistical Evaluation Report

### 6.1 Hypothesis Test Summary

| # | Test | Variables | Statistic | p-value | Decision | Effect Size |
|---|------|-----------|-----------|---------|----------|-------------|
| 1 | t-Test (Welch) | Tip: Smoker vs Non-Smoker | t | See output | Fail to reject H₀ | Cohen's d (small) |
| 2 | t-Test (Welch) | Tip: Weekend vs Weekday | t | See output | Fail to reject H₀ | Cohen's d |
| 3 | One-Way ANOVA | Tip across 4 days | F | See output | See output | η² |
| 4 | Chi-Square | Tip category × Gender | χ² | See output | Fail to reject H₀ | Cramér's V (weak) |
| 5 | Chi-Square | Smoker × Meal Time | χ² | See output | See output | Cramér's V |

### 6.2 Regression Model Evaluation

| Model | Features | Test R² | Test RMSE | Key Finding |
|-------|----------|---------|-----------|-------------|
| Simple LR | 1 (total_bill) | ~0.45 | ~1.00 | total_bill alone explains ~45% of variance |
| Multiple LR | 7 | ~0.48 | ~0.99 | Adding 6 features gives minimal gain |

### 6.3 Trend Analysis Summary

| Trend | Direction | Significance | Finding |
|-------|-----------|--------------|----------|
| Tips across days (Thur→Sun) | Slight increase | Not significant | Day label is a weak predictor |
| Tip % vs party size | Negative (declining) | Significant | Larger groups tip proportionally less |

### 6.4 Uncertainty Findings

| Metric | Finding |
|--------|---------|
| Bootstrap CIs | Non-smokers: narrower CI due to larger n — more reliable estimate |
| Coefficient of Variation | Friday lunch shows highest relative variability (small sample) |
| Residual analysis | SLR residuals are approximately normal with slight right skew |
| Confidence intervals for regression coefficients | Slope for total_bill excludes zero — reliable predictor |

---
## 7. Insight Validation & Explanation

### Insight 1 — Total bill is the dominant predictor of tip amount ✅ VALIDATED
- **Evidence:** Pearson r = 0.68 (p < 0.001), SLR R² ≈ 0.45, standardised coefficient dominates MLR
- **Mechanism:** Customers tip as a percentage of the bill; a fixed percentage creates near-linear scaling
- **Caveat:** The relationship has meaningful scatter (RMSE ≈ $1), so predictions carry real uncertainty

---

### Insight 2 — Smoking status does NOT significantly affect tip amount ✅ VALIDATED
- **Evidence:** t-test p ≥ 0.05; 95% CI for the difference includes zero; Chi-square for tip category × smoker not significant
- **Mechanism:** Any observed difference in raw averages is within the range of sampling variation
- **Implication:** Do not use smoking status as a proxy for tip prediction

---

### Insight 3 — Tip percentage decreases as party size grows ✅ VALIDATED
- **Evidence:** Pearson r (size vs tip_pct) is negative and significant; linear slope ≈ −1%/person
- **Mechanism:** Social diffusion of responsibility — in larger groups, individuals feel less personal obligation
- **Implication:** Automatic gratuity policies for parties of 5+ are statistically justified by this dataset

---

### Insight 4 — Day of week has no significant effect on tip amount ✅ VALIDATED
- **Evidence:** One-way ANOVA p ≥ 0.05; day-level linear trend not significant
- **Mechanism:** Variation within each day (driven by bill size) overwhelms variation between days
- **Implication:** Staffing decisions should be based on expected customer volume, not expected tip rates

---

### Insight 5 — Gender does not predict tip generosity category ✅ VALIDATED
- **Evidence:** Chi-square p ≥ 0.05; Cramér's V < 0.1 (negligible association)
- **Mechanism:** Male diners appear in more 'Generous' cells in absolute counts, but this reflects dataset composition (more male diners), not a higher rate of generous tipping
- **Implication:** Apparent gender-tip differences in raw data are confounded by unequal group sizes

---

### Insight 6 — Multiple regression adds minimal value over simple regression ✅ VALIDATED
- **Evidence:** R² gain < 3 percentage points despite adding 6 features
- **Mechanism:** Demographic variables (sex, smoker, time, weekend) are statistically weak predictors
- **Implication:** A parsimonious model (`tip ~ total_bill`) is preferable for real-world use; adding complexity risks overfitting without predictive benefit

---
## 8. Deliverable Summary

| Deliverable | Contents |
|-------------|----------|
| `m4_fig01_ttest.png` | Smoker vs non-smoker tips: box plot + CI plot |
| `m4_fig02_anova.png` | Violin + swarm plot by day (ANOVA) |
| `m4_fig03_chisq.png` | Tip category × gender: bar + residual heatmap |
| `m4_fig04_corr.png` | Pearson & Spearman correlation matrices |
| `m4_fig05_corrbar.png` | Ranked correlations with significance codes |
| `m4_fig06_slr.png` | SLR fit, residual plot, Q-Q plot |
| `m4_fig07_mlr.png` | MLR coefficients + actual vs predicted |
| `m4_fig08_daytrend.png` | Tip trend across days of week |
| `m4_fig09_sizetrend.png` | Tip % + bill trend across party sizes |
| `m4_fig10_bootstrap.png` | Bootstrap distributions + CI comparison |
| **Section 6** | Statistical evaluation report |
| **Section 7** | Insight validation with evidence chain |
| **This notebook** | `sprint4_milestone4.ipynb` |

---
_End of Sprint 4 — Milestone 4_